# 02 – Čišćenje i priprema podataka

U ovom notebooku provodi se čišćenje podataka i priprema varijabli za izradu modela strojnog učenja.  
Ciljevi:
- provjeriti nedostajuće vrijednosti (NaN)
- ispraviti tipove podataka (npr. `Total Charges` kao broj)
- ukloniti stupce koji nisu relevantni za predikciju
- pripremiti ciljnu varijablu za kasnije modeliranje


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

possible_paths = [
    Path("data/Telco_customer_churn.xlsx"),
    Path("../data/Telco_customer_churn.xlsx"),
    Path("/content/drive/MyDrive/csv/Telco_customer_churn.xlsx")
]

DATA_PATH = None
for p in possible_paths:
    if p.exists():
        DATA_PATH = p
        break

if DATA_PATH is None:
    raise FileNotFoundError("Dataset nije pronađen. Provjeri folder data/")

df = pd.read_excel(DATA_PATH)
df.head()



,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


## Nedostajuće vrijednosti

Prvo provjeravam koliko nedostajućih vrijednosti postoji u svakom stupcu.

In [10]:
df.isna().sum().sort_values(ascending=False)


,0
Churn Reason,5174
CustomerID,0
Count,0
State,0
Country,0
Zip Code,0
Lat Long,0
Latitude,0
City,0
Gender,0


Varijabla *Churn Reason* sadrži veliki broj nedostajućih vrijednosti jer je dostupna
samo za korisnike koji su napustili uslugu. Za korisnike koji su ostali, razlog
napuštanja ne postoji, pa je vrijednost logično prazna. Ova varijabla neće se koristiti
u modeliranju jer nije dostupna za sve korisnike.


## 3. Čišćenje stupca `Total Charges`

Stupac **Total Charges** je trenutno učitan kao tekst (`object`).
Za modeliranje mi treba da bude broj (numerički tip), zato:

1) pretvaram `Total Charges` u broj (ako postoji nevaljan tekst → postaje NaN)  
2) NaN vrijednosti zamjenjujem s 0 jer se obično odnose na nove korisnike (npr. `Tenure Months = 0`) kojima ukupni trošak još nije obračunat.


In [11]:
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce")

In [12]:
df["Total Charges"].isna().sum()


np.int64(11)

Nakon pretvorbe u numerički tip pojavilo se 11 NaN vrijednosti. Te vrijednosti popunjavam s 0 jer predstavljaju situacije gdje korisnik još nije imao naplatu.

In [13]:
df["Total Charges"] = df["Total Charges"].fillna(0)

In [14]:
df["Total Charges"].isna().sum()

np.int64(0)

## Ciljna varijabla

Cilj je predvidjeti odlazak korisnika.
U datasetu postoje dvije varijante:
- **Churn Label** (Yes/No)
- **Churn Value** (1/0)

Za modeliranje je praktičniji binarni format (0/1), pa koristim `Churn Value`.

In [15]:
df["Churn Value"].value_counts()

,count
Churn Value,
0,5174
1,1869


In [16]:
# sigurnosna provjera da se Churn Label i Churn Value poklapaju
pd.crosstab(df["Churn Label"], df["Churn Value"])


Churn Value,0,1
Churn Label,,
No,5174,0
Yes,0,1869


## Uklanjanje irelevantnih stupaca

Stupci poput identifikatora i lokacije mogu unijeti šum u model.

Uklanjam:
- CustomerID (identifikator)
- Country/State/City/Zip/Lat/Long/Latitude/Longitude (lokacija)
- Count (konstanta = 1, nema informaciju)

Napomena: `Churn Reason` je koristan za objašnjenje churn-a, ali ne smije se koristiti za predikciju jer je “post-factum” informacija (poznata tek nakon churn-a).


In [17]:
drop_cols = [
    "CustomerID", "Count",
    "Country", "State", "City", "Zip Code",
    "Lat Long", "Latitude", "Longitude",
    "Churn Reason"
]

df_clean = df.drop(columns=drop_cols, errors="ignore")
df_clean.shape

(7043, 23)

## Završna provjera

Provjeravam da nema neželjenih nedostajućih vrijednosti i da su tipovi podataka smisleni.


In [18]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Gender             7043 non-null   object 
 1   Senior Citizen     7043 non-null   object 
 2   Partner            7043 non-null   object 
 3   Dependents         7043 non-null   object 
 4   Tenure Months      7043 non-null   int64  
 5   Phone Service      7043 non-null   object 
 6   Multiple Lines     7043 non-null   object 
 7   Internet Service   7043 non-null   object 
 8   Online Security    7043 non-null   object 
 9   Online Backup      7043 non-null   object 
 10  Device Protection  7043 non-null   object 
 11  Tech Support       7043 non-null   object 
 12  Streaming TV       7043 non-null   object 
 13  Streaming Movies   7043 non-null   object 
 14  Contract           7043 non-null   object 
 15  Paperless Billing  7043 non-null   object 
 16  Payment Method     7043 

In [19]:
df_clean.isna().sum().sort_values(ascending=False)

,0
Gender,0
Senior Citizen,0
Partner,0
Dependents,0
Tenure Months,0
Phone Service,0
Multiple Lines,0
Internet Service,0
Online Security,0
Online Backup,0


## Spremi očišćeni dataset

Spremam očišćeni dataset u `data/` kao CSV.


In [24]:
from pathlib import Path

possible_paths = [
    Path("data/Telco_customer_clean.csv"),
    Path("../data/Telco_customer_clean.csv"),
    Path("/content/drive/MyDrive/csv/Telco_customer_clean.csv"),
]

DATA_PATH = next((p for p in possible_paths if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Ne mogu naći telecom_churn_clean.csv.\n"

    )

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH.resolve())


Loaded: /content/drive/MyDrive/csv/Telco_customer_clean.csv


Očišćeni dataset spremljen je u CSV format kako bi se mogao koristiti u kasnijim fazama analize i modeliranja.